In [ ]:
import torch
from confidence.utils import ModelInputOutputWrapper
from src.utils.sampling import BatchNegativeSampler


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]
budget = None
repeats = 5
if dataset == "tu_berlin":
    repeats = 20
if dataset == "modelnet10":
    repeats = 10
if dataset == "coil100":
    repeats = 10



In [ ]:
from src.utils.transforms.apply import grid_resample_border,grid_resample_reflection

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset
dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x=next(iter(test_loader_transformed))[0]

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:
from src.utils.eval.vis import vis_dataset, plt_setup_latex

vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from model.train import train_and_get_model,train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#chek if data is image data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]


from src.utils.replacer import replace_rotation_transforms_2vec


from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
from model.get_model import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 0, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence

logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq,                                              consolidate_method="consolidate_simple")
#test ot
from search.random_search import RSLR
optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search, evaluate_confidence_and_search, \
    ITSWRAPPER


In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_confidence_transformed"), show_progress=True,
    repeats=1)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#check wether main model is deterministic
x = next(iter(test_loader_transformed))[0]
out1 = model(x.to(device))
out2 = model(x.to(device))
print(torch.allclose(out1, out2))

In [ ]:
from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import src.utils.augments


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler


energy_model2 = get_network(dataset_info,architecture, num_classes=1).to(device)



if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)
namef = f"{modelname}_energy2" if not is_image_data else f"{modelname}_energy2_only_blur_aug"
energy_conf_default = train_or_load_energy_model(
    energy_model2, model_dir_path, namef, train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_default.to(device).eval()



problem_energy_default = TransformationProblem(energy_conf_default, transform_seq,
                                               consolidate_method="consolidate_simple")
savepathname = savepath("learned_energy_confidence_only_blur_aug_transformed") if is_image_data else savepath("learned_energy_confidence_transformed")
res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_default,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepathname, show_progress=True,
    repeats=repeats)


#unload values
del energy_model2
del energy_conf_default
del problem_energy_default
print(res)


In [ ]:
layer, layer_io = get_network_layer(dataset_info, architecture, 1, num_classes=None, num_rotations=8)

In [ ]:
dual_model = ModelInputOutputWrapper(model, layer, capture_modes=layer_io,flatten=False).to(device)

In [ ]:
embed,logits = dual_model(next(iter(test_loader_transformed))[0].to(device))


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddingProcessor(nn.Module):
    def __init__(self, embed_shape, embed_dim=512, conv_channels=256):
        """
        Args:
            embed_shape: shape of embeds (e.g., (B, D) or (B, C, H, W))
            embed_dim:   output feature dimension after processing
            conv_channels: number of channels in conv layers (for spatial embeds)
        """
        super().__init__()
        self.is_2d = len(embed_shape) == 2  # (B, D) = dense embeddings

        if self.is_2d:
            in_dim = embed_shape[1]
            self.network = nn.Sequential(
                nn.Linear(in_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, 1),
            )
        else:
            # assume shape = (B, C, H, W)
            c = embed_shape[1]
            self.conv_net = nn.Sequential(
                nn.Conv2d(c, conv_channels, kernel_size=3, padding=1),
                nn.GELU(),
                nn.Conv2d(conv_channels, conv_channels, kernel_size=3, padding=1),
                nn.GELU(),
                nn.AdaptiveAvgPool2d((1, 1))  # spatial pooling
            )
            self.fc = nn.Sequential(
                nn.Linear(conv_channels, embed_dim),
                nn.GELU(),
                nn.Linear(embed_dim, 1),
            )

    def forward(self, embeds):
        if self.is_2d:
            return self.network(embeds)
        else:
            x = self.conv_net(embeds)
            x = x.flatten(1)  # (B, conv_channels)
            return self.fc(x)

class DualModelLogitRemover(nn.Module):
    def __init__(self, dual_model):
        super().__init__()
        self.dual_model = dual_model

    def forward(self, x):
        embeds, _ = self.dual_model(x)
        return embeds

In [ ]:
from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import src.utils.augments

def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler

energy_model_embed = EmbeddingProcessor(embed_shape=embed.shape[:]).to(device)





negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, feature_extractor=DualModelLogitRemover(dual_model)
    ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)

energy_conf_embed = train_or_load_energy_model(
    energy_model_embed, model_dir_path, f"{modelname}_energy_embed_later_layer", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)
energy_conf_embed.to(device).eval()
problem_energy_embed = TransformationProblem(energy_conf_embed, transform_seq,                                              consolidate_method="consolidate_simple")
res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_embed,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_later_layer_transformed"), show_progress=True,
    repeats=repeats)
#unload values
print(res)


del energy_model_embed
del energy_conf_embed
del problem_energy_embed

In [ ]:
#now run the learned model directly on the predicted logits

from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma, random_blur_or_sharpen, build_default_augmentations
import src.utils.augments

def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    y = torch.where(eq, y_true, -1)
    return y

from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler




class LogitProcessor(nn.Module):
    def __init__(self, logit_shape, embed_dim=512):
        super().__init__()
        in_dim = logit_shape[1]
        self.network = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, 1),
        )
    def forward(self, logits):
        return self.network(logits)





negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, feature_extractor=model,
    ), transform_true_function=transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)

energy_model_logits = LogitProcessor(logit_shape=logits.shape[:]).to(device)

energy_conf_logits = train_or_load_energy_model(
    energy_model_logits, model_dir_path, f"{modelname}_energy_logits", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)
energy_conf_logits.to(device).eval()

problem_energy_logits = TransformationProblem(energy_conf_logits, transform_seq, consolidate_method="consolidate_simple")
res = load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_logits_transformed"), show_progress=True,
    repeats=repeats)
print(res)
del energy_model_logits
del energy_conf_logits
del problem_energy_logits


In [ ]:
from model.basic_networks import make_deterministic

# Load pretrained classifier weights into new energy model
energy_model_embed = get_network(dataset_info, architecture, num_classes=1).to(device)

# Copy matching weights from pretrained model (exclude final layer)
pretrained_state = model.state_dict()
model_state = energy_model_embed.state_dict()

filtered_state = {
    k: v for k, v in pretrained_state.items()
    if k in model_state and v.shape == model_state[k].shape
}

energy_model_embed.load_state_dict(filtered_state, strict=False)

# Freeze pretrained layers
for name, param in energy_model_embed.named_parameters():
    if name in filtered_state:
        param.requires_grad = False



# Setup decision strategy
def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    y = torch.where(eq, y_true, -1)
    return y

# Setup negative sampling
negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(transform_sequence=transform_seq),
    transform_true_function=transform_true_function,
    augment_function=affine_augment,
    decision_strategy=dec_strat
)

# Stage 1: Train only final layer (2 epochs)
energy_conf_embed = train_or_load_energy_model(
    energy_model_embed,
    model_dir_path,
    f"{modelname}_energy_embed_pretrained_stage1",
    train_loader,
    val_loader,
    trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    },
    negative_sampling_module=negative_sampling_module,
    load_if_exists=True
)
energy_conf_embed.train()
make_deterministic(energy_conf_embed,random=True)

# Stage 2: Unfreeze all layers and continue training
for param in energy_model_embed.parameters():
    param.requires_grad = True

energy_conf_embed = train_or_load_energy_model(
    energy_model_embed,
    model_dir_path,
    f"{modelname}_energy_embed_pretrained",
    train_loader,
    val_loader,
    trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2 - 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    },
    negative_sampling_module=negative_sampling_module,
    load_if_exists=True
)

energy_conf_embed.to(device).eval()

# Create transformation problem
problem_energy_embed = TransformationProblem(
    energy_conf_embed,
    transform_seq,
    consolidate_method="consolidate_simple"
)

# Evaluate
res = load_or_run_evaluate_confidence_and_search(
    model,
    optimizer=optimizer,
    problem=problem_energy_embed,
    test_loader=test_loader_transformed,
    max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_embed_pretrained_transformed"),
    show_progress=True,
    repeats=repeats
)

print(res)

In [ ]:
from typing import Optional, Any, Tuple, Dict
import math
from hyper_param.ood.base_prepare import create_ood_problem, get_default_ood_params
import os
import json
import math



In [ ]:
detectors = ["knn","per_class_knn", "knn_mixed","per_class_knn_mixed","knn_mixed_faiss","knn_itf","vim","react","dice","ash","she","laplace_mi","laplace_energy","laplace_weighted","trust_score","openmax","mahalanobis","rmd","class_prototype","react_all","energy","per_class_prototype","single_mahalanobis","single_rmd","single_mahalanobis_individual","single_rmd_individual","mahalanobis_individual","rmd_individual","prototype_multi", "laplace_entropy","adjusted_entropy","nn_guided","nn_guided_one"]

In [ ]:
from data.get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture, transform_name=None, dataset_info=dataset_info)
train_cache = create_layer_embedding_cache(model, train_loader_no_shuffle, cache_config, embedding_cache_path,
                                           device=device)

In [ ]:
train_cache

In [ ]:
base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)

In [ ]:
#plt.title("ICNN Variants on bigger_mnist")
figure_path = os.path.join(current_path,"experiment_files","export","fig","comparision_supervised_ll",dataset,transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
W=plt_setup_latex()

In [ ]:
from hyper_param.ood.base_prepare import find_best_detector_and_instantiate

best_detector, best_problem, best_score, second_choice, second_problem, second_score = find_best_detector_and_instantiate(
    base_results_dir=base_results_dir,
    detectors=detectors,
    model=model,
    train_cache=train_cache,
    transform_seq_arg=transform_seq,
    dataset_info=dataset_info,
    architecture=architecture,
    device=device,
    val_id_loader=val_loader_transformed,
    val_ood_loader=val_loader_transformed,
)

In [ ]:
base_results_dir

In [ ]:
best_score

In [ ]:
#plot the results
import json
json_orig = savepathname
data_orig = json.load(open(json_orig, "r"))

data_embed = savepath("learned_energy_confidence_later_layer_transformed")
data_embed = json.load(open(data_embed, "r"))

data_logits = savepath("learned_energy_confidence_logits_transformed")
data_logits = json.load(open(data_logits, "r"))

data_embed_pretrained = savepath("learned_energy_confidence_embed_pretrained_transformed")
data_embed_pretrained = json.load(open(data_embed_pretrained, "r"))

data_best = best_score

data_best["accuracy_mean"] = best_score["mean"]
data_best["accuracy_se"] = best_score["se"]

#plot accuracy mean and se
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
import numpy as np

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W/2, W/4.0),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    ax.set_ylabel("Accuracy")
    plt.xticks(rotation=20)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')
    plt.tight_layout()
    plt.show()


plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embed.', 'Logits', 'Unsuper.'],
    title='',
    save_path=os.path.join(figure_path, "accuracy_comparison.pgf")
)
plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embed.', 'Logits', 'Unsuper.'],
    title='',
    save_path=os.path.join(figure_path, "accuracy_comparison.pdf")
)



In [ ]:
from src.utils.eval.vis import plt_setup_paper
W = plt_setup_paper()
figure_path2 = os.path.join(current_path, "experiment_files", "export", "fig2", "comparision_supervised_ll", dataset,
                           transform_name)
os.makedirs(figure_path2, exist_ok=True)

W = plt_setup_paper()


In [ ]:
#plot the results
import json
json_orig = savepathname
data_orig = json.load(open(json_orig, "r"))

data_embed = savepath("learned_energy_confidence_later_layer_transformed")
data_embed = json.load(open(data_embed, "r"))

data_logits = savepath("learned_energy_confidence_logits_transformed")
data_logits = json.load(open(data_logits, "r"))

data_embed_pretrained = savepath("learned_energy_confidence_embed_pretrained_transformed")
data_embed_pretrained = json.load(open(data_embed_pretrained, "r"))

data_best = best_score

data_best["accuracy_mean"] = best_score["mean"]
data_best["accuracy_se"] = best_score["se"]

#plot accuracy mean and se
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W, W/4.5),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    plt.xticks(rotation=0)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')

    plt.show()



plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embeddings', 'Logits', 'Unsupervised'],
    title='',
    save_path=os.path.join(figure_path2, "accuracy_comparison2.pdf")
)



In [ ]:
#plot the results
import json
json_orig = savepathname
data_orig = json.load(open(json_orig, "r"))

data_embed = savepath("learned_energy_confidence_later_layer_transformed")
data_embed = json.load(open(data_embed, "r"))

data_logits = savepath("learned_energy_confidence_logits_transformed")
data_logits = json.load(open(data_logits, "r"))

data_embed_pretrained = savepath("learned_energy_confidence_embed_pretrained_transformed")
data_embed_pretrained = json.load(open(data_embed_pretrained, "r"))

data_best = best_score

data_best["accuracy_mean"] = best_score["mean"]
data_best["accuracy_se"] = best_score["se"]

#plot accuracy mean and se
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W/2, W/5),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    plt.xticks(rotation=18)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')

    plt.show()



plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embed', 'Logits', 'Unsupervised'],
    title='',
    save_path=os.path.join(figure_path2, "accuracy_comparison3.pdf")
)



In [ ]:
#plot the results
import json
json_orig = savepathname
data_orig = json.load(open(json_orig, "r"))

data_embed = savepath("learned_energy_confidence_later_layer_transformed")
data_embed = json.load(open(data_embed, "r"))

data_logits = savepath("learned_energy_confidence_logits_transformed")
data_logits = json.load(open(data_logits, "r"))

data_embed_pretrained = savepath("learned_energy_confidence_embed_pretrained_transformed")
data_embed_pretrained = json.load(open(data_embed_pretrained, "r"))

data_best = best_score

data_best["accuracy_mean"] = best_score["mean"]
data_best["accuracy_se"] = best_score["se"]

#plot accuracy mean and se
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W/2, W/4.0),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    ax.set_ylabel("Accuracy")
    plt.xticks(rotation=20)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')
    plt.tight_layout()
    plt.show()


plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embed.', 'Logits', 'Unsuper.'],
    title='',
    save_path=os.path.join(figure_path2, "accuracy_comparison.pdf")
)



In [ ]:
#plot the results
import json
json_orig = savepathname
data_orig = json.load(open(json_orig, "r"))

data_embed = savepath("learned_energy_confidence_later_layer_transformed")
data_embed = json.load(open(data_embed, "r"))

data_logits = savepath("learned_energy_confidence_logits_transformed")
data_logits = json.load(open(data_logits, "r"))

data_embed_pretrained = savepath("learned_energy_confidence_embed_pretrained_transformed")
data_embed_pretrained = json.load(open(data_embed_pretrained, "r"))

data_best = best_score

data_best["accuracy_mean"] = best_score["mean"]
data_best["accuracy_se"] = best_score["se"]

#plot accuracy mean and se
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np

figure_path3 = os.path.join(current_path, "experiment_files", "export", "fig3", "comparision_supervised_ll", dataset,
                           transform_name)
os.makedirs(figure_path3, exist_ok=True)
W = plt_setup_latex(W=5.53)

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W/2, W/4.0),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    ax.set_ylabel("Accuracy")
    plt.xticks(rotation=23)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')
    plt.tight_layout()
    plt.show()


plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embed.', 'Logits', 'Unsuper.'],
    title='',
    save_path=os.path.join(figure_path3, "accuracy_comparison.pdf")
)



In [ ]:

def plot_accuracy_comparison(datasets, labels, title,save_path=None):
    # Extract scalar means and standard errors
    means = np.array([d['accuracy_mean'] for d in datasets])
    ses   = np.array([d['accuracy_se']   for d in datasets]) *1.96

    x = np.arange(len(labels))

    plt.figure(figsize=(W, W/4.5),dpi=300)

    # seaborn barplot with tab10 colors
    ax = sns.barplot(
        x=labels,
        y=means,
        palette="tab10",
        ci=None,zorder=2,edgecolor='black',
        linewidth = 0.2,
    )

    # Add SE error bars
    ax.errorbar(
        x=x,
        y=means,
        yerr=ses,
        fmt='none',
        c='black',
        capsize=3,zorder=3,
        #reduce error bar width,
        lw=0.5,
        capthick=0.5,
    )

    # Dynamic y-limits
    ymin = (means - ses).min() - 0.05
    ymax = (means + ses).max() + 0.05
    ax.set_ylim(ymin, ymax)

    plt.xticks(rotation=0)
    #set font size to 8
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)


    #set label font size to 8
    ax.set_ylabel("Accuracy", fontsize=9)


    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path,bbox_inches='tight')

    plt.show()



plot_accuracy_comparison(
    datasets=[data_orig, data_embed_pretrained, data_embed, data_logits, best_score],
    labels=['Original', 'Finetune', 'Embeddings', 'Logits', 'Unsupervised'],
    title='',
    save_path=os.path.join(figure_path2, "accuracy_comparison2.pdf")
)
